# 12 · Update, upsert and delete safely
Level L2 · Part 2 · builds on lesson 11 (and lessons 09, 10)

## Goal
By the end of this lesson you can change a recipe's payload, replace its vector and delete
recipes (one by id, many by filter) so that the next search shows the change and nothing else is
lost.

## The idea
Recipes change. The Butternut squash soup now takes 25 minutes instead of 45, and Gazpacho comes
off the menu. A vector database gives you a different tool for each kind of change. **Upsert**
replaces the whole point. **Set payload** changes only the fields you name. **Update vectors**
swaps only the vector. **Delete** removes points by id or by filter. Choosing the wrong tool rarely
raises an error: you silently lose fields, keep a stale vector, or delete too much. Think of
editing a shared recipe card: you can rewrite the whole card, cross out one line, or throw the card
away, and two cooks editing the same card at once need a rule for who wins.

## Picture

```mermaid
flowchart LR
    P["point 11<br/>id + vector + payload"]
    P -->|"upsert (whole point)"| A["new vector + new payload<br/>(fields you leave out are gone)"]
    P -->|"set_payload"| B["same vector<br/>named fields changed, others kept"]
    P -->|"update_vectors"| C["new vector<br/>payload kept"]
    P -->|"delete"| D["gone from search,<br/>count and scroll"]
```

Each write can also carry a condition (`update_filter`) or a mode (`update_mode`): "only if the
version is still 2", "only if the point already exists".

## Step by step
### Step 1 · The `recipes` collection, with a version number
Rebuild lesson 11's collection in this lesson's own folder. The payload gets one new field,
`version`, starting at 1. Step 7 uses it to stop two edits from overwriting each other.

In [1]:
import shutil, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", message="IProgress")     # quiet a progress-bar notice on import
from qdrant_client import QdrantClient, models

recipes = pd.read_csv("../data/recipes_50.csv")
saved = np.load("../data/recipe_embeddings_minilm.npz")
doc_emb, MODEL = saved["doc_emb"], str(saved["model"])
print("model:", MODEL, "| recipes:", doc_emb.shape)

model: sentence-transformers/all-MiniLM-L6-v2 | recipes: (50, 384)


`to_payload` is lesson 11's helper plus `version`. `load_recipes` is lesson 09's idempotent load:
one point per CSV row, keyed by the recipe id.

In [2]:
DB_PATH = "../data/cache/qdrant_12"
shutil.rmtree(DB_PATH, ignore_errors=True)           # start from an empty folder every run
client = QdrantClient(path=DB_PATH)
client.create_collection("recipes", vectors_config=models.VectorParams(
    size=doc_emb.shape[1], distance=models.Distance.COSINE))

def to_payload(row):
    return {"title": str(row.title), "description": str(row.description),
            "cuisine": str(row.cuisine), "vegetarian": bool(row.vegetarian),
            "minutes": int(row.minutes), "embed_model": MODEL, "version": 1}

def load_recipes(**kwargs):
    points = [models.PointStruct(id=int(row.id), vector=doc_emb[i].tolist(),
                                 payload=to_payload(row)) for i, row in enumerate(recipes.itertuples())]
    return client.upsert("recipes", points=points, **kwargs)

print(load_recipes())
print("points:", client.count("recipes").count)
print(client.retrieve("recipes", ids=[11])[0].payload)

operation_id=0 status=<UpdateStatus.COMPLETED: 'completed'>
points: 50
{'title': 'Butternut squash soup', 'description': 'Roasted squash blended with ginger and coconut milk into a velvety soup.', 'cuisine': 'Thai', 'vegetarian': True, 'minutes': 45, 'embed_model': 'sentence-transformers/all-MiniLM-L6-v2', 'version': 1}


The update needs two new pieces of text: the soup's **new description** (Step 6) and the query "a
quick vegetarian soup".

In [3]:
NEW_SOUP_TEXT = ("Quick version: microwave the squash, then blend with ginger and coconut milk. "
                 "Ready in 25 minutes.")
SOUP_QUERY = "a quick vegetarian soup"
print(NEW_SOUP_TEXT, "|", SOUP_QUERY)

Quick version: microwave the squash, then blend with ginger and coconut milk. Ready in 25 minutes. | a quick vegetarian soup


The next cell embeds both with the same model and saves them (it needs the model download; the
saved file ships in `data/`, so the cell after it works without it). The recipe text is embedded
as `title + ". " + description`, like every other recipe.

In [4]:
import os
os.environ.update(TQDM_DISABLE="1", HF_HUB_VERBOSITY="error")
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(MODEL)
soup_text = "Butternut squash soup. " + NEW_SOUP_TEXT
np.savez("../data/lesson12_embeddings_minilm.npz", model=MODEL, doc_text=np.array([soup_text]),
         doc_emb=model.encode_document([soup_text]),
         queries=np.array([SOUP_QUERY]), query_emb=model.encode_query([SOUP_QUERY]))
print("saved 1 recipe vector and 1 query vector")

⏭  Skipped: this cell needs [download]. Run it yourself, or re-run with TUT_ENABLE=download.


Load the saved vectors and check they came from the same model as the collection:

In [5]:
extra = np.load("../data/lesson12_embeddings_minilm.npz")
assert str(extra["model"]) == MODEL
new_soup_vector = extra["doc_emb"][0].tolist()
soup_query = extra["query_emb"][0].tolist()
print("new recipe text:", str(extra["doc_text"][0]))
print("vector lengths:", len(new_soup_vector), len(soup_query))

new recipe text: Butternut squash soup. Quick version: microwave the squash, then blend with ginger and coconut milk. Ready in 25 minutes.
vector lengths: 384 384


### Step 2 · The search before any change
Lesson 11's "quick vegetarian" filter, used for the soup query. `show` prints a page of results;
we call it after every change to see what search now returns.

In [6]:
quick_veg = models.Filter(must=[
    models.FieldCondition(key="vegetarian", match=models.MatchValue(value=True)),
    models.FieldCondition(key="minutes", range=models.Range(lte=30)),
])

def show(query=soup_query, flt=quick_veg, k=5):
    hits = client.query_points("recipes", query=query, query_filter=flt, limit=k).points
    for h in hits:
        print(f"  {h.id:>3} {h.payload['title']:<24} {h.payload['minutes']:>3} min  {h.score:.3f}")

show()
print("quick vegetarian recipes:", client.count("recipes", count_filter=quick_veg).count)

   50 Gazpacho                  15 min  0.600
   47 Hummus                    10 min  0.498
    2 Mango salsa               10 min  0.491
   27 Spicy tofu stir-fry       20 min  0.461
   46 Guacamole                 10 min  0.458
quick vegetarian recipes: 25


Gazpacho leads at 0.600. The Butternut squash soup is not on the page: at 45 minutes it fails the
`minutes <= 30` rule. 25 recipes pass the filter.

### Step 3 · The trap: upsert with only the changed field
Lesson 09 said upsert replaces the **whole** point. See what that means when you "just" want to
change the minutes:

In [7]:
client.upsert("recipes", points=[models.PointStruct(
    id=11, vector=doc_emb[10].tolist(), payload={"minutes": 25})])
print(client.retrieve("recipes", ids=[11])[0].payload)

{'minutes': 25}


The payload is now only `{'minutes': 25}`. Title, cuisine, `vegetarian` and the rest are gone, so
the soup fails the `vegetarian` rule and has no title to show. No error was raised. Repair it by
re-running the full load, which is safe because it is idempotent (lesson 09):

In [8]:
load_recipes()
print(client.retrieve("recipes", ids=[11])[0].payload)

{'title': 'Butternut squash soup', 'description': 'Roasted squash blended with ginger and coconut milk into a velvety soup.', 'cuisine': 'Thai', 'vegetarian': True, 'minutes': 45, 'embed_model': 'sentence-transformers/all-MiniLM-L6-v2', 'version': 1}


### Step 4 · `set_payload`: change only the named fields
`set_payload` writes the fields you give and keeps every other field and the vector. The docs call
it "Set only the given payload values on a point." `points` takes a list of ids. We also raise
`version` to 2, because the recipe changed.

In [9]:
result = client.set_payload("recipes", payload={"minutes": 25, "version": 2}, points=[11])
print(result)
print(client.retrieve("recipes", ids=[11])[0].payload)

operation_id=0 status=<UpdateStatus.COMPLETED: 'completed'>
{'title': 'Butternut squash soup', 'description': 'Roasted squash blended with ginger and coconut milk into a velvety soup.', 'cuisine': 'Thai', 'vegetarian': True, 'minutes': 25, 'embed_model': 'sentence-transformers/all-MiniLM-L6-v2', 'version': 2}


Every other field survived. The client waits by default (`wait=True`), so the change is applied
when the call returns.

Search immediately reflects the new payload:

In [10]:
show()
print("quick vegetarian recipes:", client.count("recipes", count_filter=quick_veg).count)

   11 Butternut squash soup     25 min  0.630
   50 Gazpacho                  15 min  0.600
   47 Hummus                    10 min  0.498
    2 Mango salsa               10 min  0.491
   27 Spicy tofu stir-fry       20 min  0.461
quick vegetarian recipes: 26


The soup now passes the filter and goes straight to the top at 0.630. 26 recipes pass instead of
25. No re-indexing step, no reload.

### Step 5 · Four payload operations side by side
There are four payload writes. To compare them, apply each one to a fresh copy of the same payload
(on a throwaway point, id 900):

In [11]:
start = {"title": "Test soup", "minutes": 45, "vegetarian": True, "version": 1}
operations = {
    "set_payload({'minutes': 25})": lambda: client.set_payload(
        "recipes", payload={"minutes": 25}, points=[900]),
    "overwrite_payload({'minutes': 25})": lambda: client.overwrite_payload(
        "recipes", payload={"minutes": 25}, points=[900]),
    "delete_payload(['version'])": lambda: client.delete_payload(
        "recipes", keys=["version"], points=[900]),
    "clear_payload()": lambda: client.clear_payload("recipes", points_selector=[900]),
}
for name, operation in operations.items():
    client.upsert("recipes", points=[models.PointStruct(id=900, vector=doc_emb[0].tolist(),
                                                        payload=start)])
    operation()
    print(f"{name:<36} -> {client.retrieve('recipes', ids=[900])[0].payload}")
client.delete("recipes", points_selector=[900])
print("points:", client.count("recipes").count)

set_payload({'minutes': 25})         -> {'title': 'Test soup', 'minutes': 25, 'vegetarian': True, 'version': 1}
overwrite_payload({'minutes': 25})   -> {'minutes': 25}
delete_payload(['version'])          -> {'title': 'Test soup', 'minutes': 45, 'vegetarian': True}
clear_payload()                      -> {}
points: 50


`set_payload` merges. `overwrite_payload` replaces the whole payload (like Step 3's upsert, but
the vector stays). `delete_payload` removes the named keys. `clear_payload` removes them all. We
deleted the test point, so the collection is back to 50.

### Step 6 · Replace the vector when the text changes
The chef also rewrote the description. The **vector was made from the old text**, so search still
"reads" the old recipe. Update the description with `set_payload`, then compare the stored vector
with the new text's vector:

In [12]:
client.set_payload("recipes", payload={"description": NEW_SOUP_TEXT, "version": 3}, points=[11])
stored = client.retrieve("recipes", ids=[11], with_vectors=True)[0]
print("description:", stored.payload["description"])
print("stored vector vs new text:", round(float(np.dot(stored.vector, new_soup_vector)), 3))

description: Quick version: microwave the squash, then blend with ginger and coconut milk. Ready in 25 minutes.
stored vector vs new text: 0.886


A cosine of 0.886: close, but not the vector of the new text. Fix it with
`update_vectors`. It takes `PointVectors(id, vector)` and keeps the payload:

In [13]:
client.update_vectors("recipes", points=[models.PointVectors(id=11, vector=new_soup_vector)])
stored = client.retrieve("recipes", ids=[11], with_vectors=True)[0]
print("stored vector vs new text:", round(float(np.dot(stored.vector, new_soup_vector)), 3))
print("payload kept:", stored.payload["title"], stored.payload["minutes"], "min, version",
      stored.payload["version"])
show()

stored vector vs new text: 1.0
payload kept: Butternut squash soup 25 min, version 3
   50 Gazpacho                  15 min  0.600
   11 Butternut squash soup     25 min  0.596
   47 Hummus                    10 min  0.498
    2 Mango salsa               10 min  0.491
   27 Spicy tofu stir-fry       20 min  0.461


The stored vector now matches the new text (1.0). The soup's score for "a quick vegetarian soup"
drops a little, from 0.630 to 0.596, and Gazpacho is first again. That is correct: search now
ranks the text the chef actually wrote. Rule: **whenever the embedded text changes, re-embed and
update the vector in the same change**, with the same model as the rest of the collection.

### Step 7 · Conditional updates: don't overwrite someone else's change
Two editors open the soup at the same moment. Both read `version` 3. Editor A sets 30 minutes;
editor B, a moment later, sets 20 minutes. Without a rule, the last write wins and A's change is
lost. A **conditional update** passes `update_filter`: the write is applied only
to points that still match it. Here: "only if `version` is still the one I read".

One catch: the docs warn that "a conditional update on a non-existent point behaves as a regular
upsert, inserting the point regardless of the filter". So the helper also passes
`update_mode=UPDATE_ONLY` (Qdrant 1.17): "Points that do not exist are not inserted".

In [14]:
def save_if_version(point_id, changes, expected_version):
    current = client.retrieve("recipes", ids=[point_id], with_vectors=True)[0]
    payload = {**current.payload, **changes, "version": expected_version + 1}
    client.upsert("recipes",
                  points=[models.PointStruct(id=point_id, vector=current.vector, payload=payload)],
                  update_filter=models.Filter(must=[models.FieldCondition(
                      key="version", match=models.MatchValue(value=expected_version))]),
                  update_mode=models.UpdateMode.UPDATE_ONLY)
    after = client.retrieve("recipes", ids=[point_id])[0].payload
    applied = after["version"] == expected_version + 1 and all(
        after[k] == v for k, v in changes.items())
    return applied, after

print("save_if_version: writes only if 'version' still equals the value you read")

save_if_version: writes only if 'version' still equals the value you read


The helper writes, then reads the point back to see whether *its* write landed: the version went
up by one and the fields hold its values. (It reads the vector only so the upsert can resend it
unchanged.) Both editors read version 3:

In [15]:
seen_by_a = seen_by_b = client.retrieve("recipes", ids=[11])[0].payload["version"]
ok_a, after_a = save_if_version(11, {"minutes": 30}, seen_by_a)
print("editor A applied:", ok_a, "| minutes", after_a["minutes"], "| version", after_a["version"])
ok_b, after_b = save_if_version(11, {"minutes": 20}, seen_by_b)
print("editor B applied:", ok_b, "| minutes", after_b["minutes"], "| version", after_b["version"])

editor A applied: True | minutes 30 | version 4
editor B applied: False | minutes 30 | version 4


A's write matched `version == 3`, so it was applied and the version became 4. B's write still
expected 3, matched nothing, and was **skipped**: minutes stays 30. Qdrant raises no error for a
skipped conditional write (the upsert still reports `completed`), which is why the helper reads
the point back. B should now re-read version 4 and decide again. This pattern is called
**optimistic concurrency control**. The chef settles on 25 minutes:

In [16]:
ok, after = save_if_version(11, {"minutes": 25}, after_b["version"])
print("applied:", ok, "| minutes", after["minutes"], "| version", after["version"])

applied: True | minutes 25 | version 5


Now the proof that `update_only` never creates a point. A late edit arrives for a draft recipe
(id 51) that was never published:

In [17]:
draft = models.PointStruct(id=51, vector=doc_emb[0].tolist(), payload={"title": "Draft soup"})
client.upsert("recipes", points=[draft], update_mode=models.UpdateMode.UPDATE_ONLY)
print("with update_only:", client.retrieve("recipes", ids=[51]), "| points:",
      client.count("recipes").count)

with update_only: [] | points: 50


Nothing was created, and the count is still 50.

In the Python client, `update_filter` is a parameter of `upsert` and `update_vectors` only; for
payload writes, use a filter as the point selector (Step 9). The filter is checked per point, so
one batch can partly apply. Two test points, versions 1 and 2, get the same write "only if
version is 1":

In [18]:
test_pair = lambda title: [models.PointStruct(id=i, vector=doc_emb[0].tolist(),
                                              payload={"title": title, "version": v})
                           for i, v in [(900, 1), (901, 2)]]
client.upsert("recipes", points=test_pair("original"))
client.upsert("recipes", points=test_pair("edited"), update_filter=models.Filter(must=[
    models.FieldCondition(key="version", match=models.MatchValue(value=1))]))
for p in client.retrieve("recipes", ids=[900, 901]):
    print(p.id, p.payload)
client.delete("recipes", points_selector=[900, 901])
print("points:", client.count("recipes").count)

900 {'title': 'edited', 'version': 1}
901 {'title': 'original', 'version': 2}
points: 50


Only point 900 (version 1) was edited; 901 kept "original". Deleting both test points brings the
count back to 50.

### Step 8 · Delete one recipe by id
Gazpacho (id 50) comes off the menu. `delete` takes a list of ids:

In [19]:
print(client.delete("recipes", points_selector=[50]))
print("retrieve 50:", client.retrieve("recipes", ids=[50]))
print("points:", client.count("recipes").count)
show()

operation_id=0 status=<UpdateStatus.COMPLETED: 'completed'>
retrieve 50: []
points: 49
   11 Butternut squash soup     25 min  0.596
   47 Hummus                    10 min  0.498
    2 Mango salsa               10 min  0.491
   27 Spicy tofu stir-fry       20 min  0.461
   46 Guacamole                 10 min  0.458


Gone from `retrieve`, from `count` (49) and from search: the soup is first, and Guacamole moves
up to fill the page. Deleting an id that does not exist is also silent, so check with `retrieve`
when it matters. The CSV still lists Gazpacho, so a plain re-load (or any loader that adds missing
ids) would re-create it. Remove deleted recipes from the source too.

### Step 9 · Delete many by filter, safely
The restaurant drops its French menu. `delete` also accepts a filter, which removes **every**
matching point. That is powerful and easy to get wrong, so do it in two safe stages. First a
**soft delete**: mark the points with `set_payload`, giving a filter instead of ids, and exclude
marked points from search with `must_not`.

In [20]:
french = models.Filter(must=[
    models.FieldCondition(key="cuisine", match=models.MatchValue(value="French"))])
print("French recipes to retire:", client.count("recipes", count_filter=french).count)
client.set_payload("recipes", payload={"retired": True}, points=french)
retired = models.FieldCondition(key="retired", match=models.MatchValue(value=True))
live = models.Filter(must_not=[retired])
print("live recipes:", client.count("recipes", count_filter=live).count)

French recipes to retire: 5


live recipes: 44


5 French recipes are marked; 44 are live. Any search must now add `must_not=[retired]`. Lemon
sorbet (20 min) and Vanilla ice cream (30 min) are French, vegetarian and quick, so the quick
vegetarian filter needs the rule too:

In [21]:
quick_veg_live = models.Filter(must=quick_veg.must, must_not=[retired])
print("quick veg:", client.count("recipes", count_filter=quick_veg).count,
      "| quick veg, live only:", client.count("recipes", count_filter=quick_veg_live).count)
show(flt=quick_veg_live)

quick veg: 25 | quick veg, live only: 23
   11 Butternut squash soup     25 min  0.596
   47 Hummus                    10 min  0.498
    2 Mango salsa               10 min  0.491
   27 Spicy tofu stir-fry       20 min  0.461
   46 Guacamole                 10 min  0.458


Two quick vegetarian French recipes are hidden (25 to 23). A soft delete can be undone with
`delete_payload(["retired"])`. Once everyone agrees, **purge**: count with the exact filter you will
delete with, delete, and check the count dropped by that much.

In [22]:
purge = models.Filter(must=[retired])
before, to_delete = client.count("recipes").count, client.count("recipes", count_filter=purge).count
print("about to delete:", to_delete, "of", before)
client.delete("recipes", points_selector=models.FilterSelector(filter=purge))
after = client.count("recipes").count
print("after:", after, "| dropped by", before - after, "| as expected:", before - after == to_delete)

about to delete: 5 of 49
after: 44 | dropped by 5 | as expected: True


49 to 44, exactly the 5 we counted. `points_selector` takes a `FilterSelector(filter=...)` (a
bare `Filter` also works).

## What just happened
- Step 3: an upsert with only `{'minutes': 25}` wiped every other field, with no error.
- Step 4: `set_payload` changed minutes and version only; the soup jumped to the top of the quick
  vegetarian page (0.630) and the filter count went from 25 to 26.
- Step 6: after the description changed, the stored vector no longer matched the text; after
  `update_vectors` it did (1.0) and the score moved to 0.596.
- Step 7: of two edits made from version 3, the first applied and the second was skipped by
  `update_filter`, silently, so the helper read the point back to tell. `update_only` created
  nothing for the missing id 51, and a 2-point batch with a filter changed only the matching point.
- Step 8: deleting Gazpacho by id removed it from `retrieve`, `count` (49) and search.
- Step 9: a soft delete hid 5 French recipes; the purge counted 5, deleted 5, left 44.

## Common mistakes

**1. Passing a single id instead of a list.** `delete`, `set_payload` and friends take a
*selector*: a list of ids or a filter.

In [23]:
try:
    client.delete("recipes", points_selector=11)
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

ValueError: Unsupported selector type: <class 'int'>


It prints `ValueError: Unsupported selector type: <class 'int'>`. Fix: `points_selector=[11]`.
Nothing was deleted, as the count confirms:

In [24]:
print("points:", client.count("recipes").count, "| soup still there:",
      client.retrieve("recipes", ids=[11])[0].payload["title"])

points: 44 | soup still there: Butternut squash soup


**2. Deleting with an empty filter.** A `Filter()` with no conditions has no rule to fail, so it
matches **every** point. If the conditions come from user input or a config file and that list
turns out to be empty, a "delete by filter" empties the collection, with no error. See it on a
throwaway in-memory copy:

In [25]:
scratch = QdrantClient(":memory:")
scratch.create_collection("recipes", vectors_config=models.VectorParams(
    size=doc_emb.shape[1], distance=models.Distance.COSINE))
scratch.upsert("recipes", points=[models.PointStruct(id=int(r.id), vector=doc_emb[i].tolist())
                                  for i, r in enumerate(recipes.itertuples())])
conditions = []                                      # e.g. a cuisine list that came back empty
print("would delete:", scratch.count("recipes", count_filter=models.Filter(must=conditions)).count)
scratch.delete("recipes", points_selector=models.Filter(must=conditions))
print("left:", scratch.count("recipes").count)

would delete: 50
left: 0


Count said 50 would go, and 0 are left. Fix: refuse to delete when there are no conditions, and
compare the count with what you expect (Step 9's two-step purge):

In [26]:
def safe_delete(c, flt, expected):
    if not (flt.must or flt.should or flt.must_not):
        raise ValueError("refusing to delete with an empty filter")
    n = c.count("recipes", count_filter=flt).count
    if n != expected:
        raise ValueError(f"filter matches {n} points, expected {expected}")
    c.delete("recipes", points_selector=models.FilterSelector(filter=flt))
    return n

mango = models.Filter(must=[
    models.FieldCondition(key="title", match=models.MatchValue(value="Mango salsa"))])
print("deleted:", safe_delete(client, mango, expected=1))
print("points:", client.count("recipes").count)
scratch.close()

deleted: 1
points: 43


The guarded delete removed exactly one recipe (Mango salsa), leaving 43.

## Try it
1. **Back to slow.** The quick version did not work out. Using `save_if_version`, set the soup
   back to 45 minutes, passing the version you just read. Then call it again with the old version
   number and show it is skipped. Finally run `show()` to confirm the soup left the quick page.
2. **Retire the chocolate.** Soft-delete every recipe whose description mentions "chocolate"
   (full-text match, lesson 11), print how many are live, then purge them with `safe_delete` and
   print the final count.

In [27]:
# your code here

Close the client so the local folder is released.

In [28]:
client.close()
print("closed")

closed


## Key terms
- **set payload** — `set_payload`: write the named payload fields, keep the others and the vector.
- **overwrite payload** — `overwrite_payload`: replace the whole payload, keep the vector.
- **delete payload / clear payload** — remove the named keys / all keys from the payload.
- **update vectors** — `update_vectors` with `PointVectors(id, vector)`: replace a point's vector,
  keep its payload.
- **conditional update** — a write with `update_filter`: applied only to points that still match
  the filter; non-matching points are skipped silently.
- **optimistic concurrency control** — read a version, write only if the version is unchanged,
  and re-read on conflict.
- **update mode** — `update_mode` on upsert; `update_only` skips ids that do not exist, so a
  conditional write never creates a point (`insert_only` comes in lesson 14).
- **delete by filter** — `delete` with a filter selector: removes every matching point; an empty
  filter matches all.
- **soft delete** — mark points (e.g. `retired: true`) and exclude them in queries before
  deleting them for real.